In [ ]:
"""
TinyLlama Round 2 — Sequential Fine-tuning on MentalChat16K
Fixes first-person confusion from Round 1 (EmpatheticDialogues)
Dataset: ShenLab/MentalChat16K — 16K empathetic counseling pairs

Run: python train_round2.py
"""

import os
import sys
import torch
import logging
import random
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    EarlyStoppingCallback,
    TrainerCallback,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
    TaskType,
)
from trl import SFTTrainer
from transformers import DataCollatorForLanguageModeling


In [ ]:
# Force unbuffered output
os.environ["PYTHONUNBUFFERED"] = "1"
sys.stdout.reconfigure(line_buffering=True)

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
CONFIG = {
    "round1_model_dir":  "./tinyllama-emotional-final",
    "base_model":        "TinyLlama/TinyLlama-1.1B-Chat-v1.0",

    "output_dir":        "./tinyllama-round2-checkpoints",
    "final_model_dir":   "./tinyllama-emotional-r2-final",
    "merged_model_dir":  "./tinyllama-emotional-r2-merged",
    "backup_dir":        "./tinyllama-round2-backup",

    "max_seq_length":    512,
    "learning_rate":     3e-5,       # higher than before — 1e-5 was too low
    "num_epochs":        2,          # 2 epochs — more learning time
    "batch_size":        4,
    "grad_accum":        8,          # effective batch = 32
    "eval_steps":        50,
    "save_steps":        50,         # must match eval_steps
    "lora_r":            16,
    "lora_alpha":        32,
    "lora_dropout":      0.05,
    "patience":          4,
    "log_file":          "training_round2.log",
    "seed":              42,
}


In [ ]:
# LOGGING
# ─────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(CONFIG["log_file"]),
        logging.StreamHandler(sys.stdout)
    ]
)
log = logging.getLogger(__name__)



In [ ]:
# ─────────────────────────────────────────────
# SYSTEM PROMPT
# ─────────────────────────────────────────────
SYSTEM_PROMPT = (
    "You are a warm, empathetic AI companion. "
    "Always respond TO the user, never AS the user. "
    "Use simple, natural, conversational language. "
    "Never use formal, legal, or clinical language. "
    "Never say: I felt, I had, I got, I remember, I also, I once. "
    "Never share personal experiences — you are an AI. "
    "Focus entirely on understanding and supporting the user. "
    "Keep responses concise and ask one thoughtful follow-up question."
)


# ─────────────────────────────────────────────
# STEP 1 — GPU CHECK
# ─────────────────────────────────────────────
def check_gpu():
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU found. This script requires a CUDA GPU.")
    log.info(f"GPU   : {torch.cuda.get_device_name(0)}")
    log.info(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    log.info(f"Torch : {torch.__version__}")


In [ ]:
# ─────────────────────────────────────────────
# STEP 2 — EMOTION DETECTION
# ─────────────────────────────────────────────
def detect_emotion(text):
    text = text.lower()
    if any(w in text for w in ["happy", "excited", "great", "amazing",
                                "joy", "wonderful", "thrilled", "proud"]):
        return "joy"
    if any(w in text for w in ["sad", "depress", "cry", "lonely",
                                "hopeless", "grief", "loss", "empty",
                                "numb", "worthless", "miss"]):
        return "sadness"
    if any(w in text for w in ["angry", "anger", "furious", "rage",
                                "frustrated", "irritated", "annoyed"]):
        return "anger"
    if any(w in text for w in ["scared", "afraid", "anxious", "anxiety",
                                "fear", "terror", "panic", "worry",
                                "overwhelm", "stress", "nervous"]):
        return "fear"
    if any(w in text for w in ["disgust", "sick", "gross", "hate", "repuls"]):
        return "disgust"
    if any(w in text for w in ["surprise", "shock", "unexpected", "sudden"]):
        return "surprise"
    return "neutral"

In [ ]:
# ─────────────────────────────────────────────
# STEP 3 — LOAD MENTALCHAT16K
# ─────────────────────────────────────────────
def load_data(tokenizer):
    log.info("Loading ShenLab/MentalChat16K...")

    try:
        ds = load_dataset(
            "ShenLab/MentalChat16K",
            split="train",
            trust_remote_code=True
        )
        log.info(f"Loaded   : {len(ds)} rows")
        log.info(f"Fields   : {ds.column_names}")

    except Exception as e:
        raise RuntimeError(f"Failed to load ShenLab/MentalChat16K: {e}")

    pairs   = []
    skipped = 0

    for row in ds:
        user = str(row.get("input",  "")).strip()
        bot  = str(row.get("output", "")).strip()

        # Quality filters
        if not user or not bot:
            skipped += 1
            continue
        if len(bot) < 30:
            skipped += 1
            continue
        if len(bot) > 1000:
            bot = bot[:1000]
        if len(user) > 500:
            user = user[:500]

        emotion = detect_emotion(user)

        prompt = (
            f"<|system|>\n{SYSTEM_PROMPT}\n"
            f"<|user|>\n[Emotion: {emotion}] {user}\n"
            f"<|assistant|>\n{bot}{tokenizer.eos_token}"
        )
        pairs.append({"text": prompt})

    log.info(f"Valid pairs : {len(pairs)}")
    log.info(f"Skipped     : {skipped}")

    # Shuffle
    random.seed(CONFIG["seed"])
    random.shuffle(pairs)

    # 90/10 split
    split_idx   = int(len(pairs) * 0.9)
    train_pairs = pairs[:split_idx]
    val_pairs   = pairs[split_idx:]

    log.info(f"Train : {len(train_pairs)}")
    log.info(f"Val   : {len(val_pairs)}")
    log.info("\nSample prompt:\n" + train_pairs[0]["text"][:400])

    return Dataset.from_list(train_pairs), Dataset.from_list(val_pairs)

In [ ]:
# ─────────────────────────────────────────────
# STEP 4 — LOAD ROUND 1 MODEL
# ─────────────────────────────────────────────
def load_round1_model():
    log.info(f"Loading Round 1 from: {CONFIG['round1_model_dir']}")

    if not os.path.exists(CONFIG["round1_model_dir"]):
        raise FileNotFoundError(
            f"Round 1 model not found at {CONFIG['round1_model_dir']}.\n"
            f"Run train_tinyllama.py first.\n"
            f"Current dir contents: {os.listdir('.')}"
        )

    tokenizer = AutoTokenizer.from_pretrained(CONFIG["base_model"])
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.padding_side = "right"

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    base_model = AutoModelForCausalLM.from_pretrained(
        CONFIG["base_model"],
        quantization_config=bnb_config,
        device_map={"": 0},
        torch_dtype=torch.float16,
    )

    model = PeftModel.from_pretrained(
        base_model,
        CONFIG["round1_model_dir"],
        local_files_only=True,
        is_trainable=True,
    )

    model.config.use_cache      = False
    model.config.pretraining_tp = 1

    log.info("Round 1 model loaded ✅")
    return model, tokenizer

In [ ]:
# ─────────────────────────────────────────────
# STEP 5 — APPLY ROUND 2 LORA
# ─────────────────────────────────────────────
def apply_round2_lora(model):
    log.info("Applying Round 2 LoRA...")
    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=CONFIG["lora_r"],
        lora_alpha=CONFIG["lora_alpha"],
        target_modules=[
            "q_proj", "k_proj",
            "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj"
        ],
        lora_dropout=CONFIG["lora_dropout"],
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model

In [ ]:
# ─────────────────────────────────────────────
# STEP 6 — CHECKPOINT BACKUP CALLBACK
# ─────────────────────────────────────────────
class CheckpointBackupCallback(TrainerCallback):
    def __init__(self, backup_dir):
        self.backup_dir = backup_dir
        os.makedirs(backup_dir, exist_ok=True)

    def on_save(self, args, state, control, **kwargs):
        import shutil
        src = f"{args.output_dir}/checkpoint-{state.global_step}"
        dst = f"{self.backup_dir}/checkpoint-{state.global_step}"
        if os.path.exists(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
            log.info(f"Backed up checkpoint-{state.global_step} ✅")

In [ ]:
# ─────────────────────────────────────────────
# STEP 7 — TRAINING
# ─────────────────────────────────────────────
def train(model, tokenizer, train_dataset, val_dataset):
    log.info("Setting up trainer...")

    training_args = TrainingArguments(
        output_dir=CONFIG["output_dir"],
        num_train_epochs=CONFIG["num_epochs"],
        per_device_train_batch_size=CONFIG["batch_size"],
        per_device_eval_batch_size=CONFIG["batch_size"],
        gradient_accumulation_steps=CONFIG["grad_accum"],
        gradient_checkpointing=True,
        optim="paged_adamw_8bit",
        learning_rate=CONFIG["learning_rate"],
        weight_decay=0.001,
        fp16=True,
        bf16=False,
        max_grad_norm=0.3,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        evaluation_strategy="steps",
        eval_steps=CONFIG["eval_steps"],
        save_strategy="steps",
        save_steps=CONFIG["save_steps"],
        save_total_limit=3,
        logging_steps=10,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        report_to="none",
        group_by_length=True,
        seed=CONFIG["seed"],
    )

    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
        dataset_text_field="text",
        max_seq_length=CONFIG["max_seq_length"],
        packing=False,
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=CONFIG["patience"],
                early_stopping_threshold=0.005,
            ),
            CheckpointBackupCallback(CONFIG["backup_dir"]),
        ]
    )

    log.info(f"Train samples : {len(train_dataset)}")
    log.info(f"Val samples   : {len(val_dataset)}")
    log.info(f"Learning rate : {CONFIG['learning_rate']}")
    log.info(f"Epochs        : {CONFIG['num_epochs']}")
    log.info("Starting Round 2 training...")

    trainer.train()
    log.info("Round 2 training complete ✅")
    return trainer

In [ ]:
# ─────────────────────────────────────────────
# STEP 8 — SAVE & MERGE R1 + R2
# ─────────────────────────────────────────────
def save_and_merge(model, tokenizer):
    log.info(f"Saving R2 adapter → {CONFIG['final_model_dir']}")
    model.save_pretrained(CONFIG["final_model_dir"])
    tokenizer.save_pretrained(CONFIG["final_model_dir"])
    log.info("R2 adapter saved ✅")

    log.info("Merging R1 + R2 into base model...")

    base_model = AutoModelForCausalLM.from_pretrained(
        CONFIG["base_model"],
        torch_dtype=torch.float16,
        device_map="cpu",
    )

    log.info("Merging Round 1...")
    merged = PeftModel.from_pretrained(
        base_model,
        CONFIG["round1_model_dir"],
        local_files_only=True,
    )
    merged = merged.merge_and_unload()

    log.info("Merging Round 2...")
    merged = PeftModel.from_pretrained(
        merged,
        CONFIG["final_model_dir"],
        local_files_only=True,
    )
    merged = merged.merge_and_unload()

    os.makedirs(CONFIG["merged_model_dir"], exist_ok=True)
    merged.save_pretrained(CONFIG["merged_model_dir"], safe_serialization=True)
    tokenizer.save_pretrained(CONFIG["merged_model_dir"])
    log.info(f"Final merged model → {CONFIG['merged_model_dir']} ✅")


In [ ]:
# ─────────────────────────────────────────────
# STEP 9 — INFERENCE TEST
# ─────────────────────────────────────────────
def chat(model, tokenizer, user_input, emotion=None,
         max_new_tokens=120, temperature=0.7):

    tag    = f"[Emotion: {emotion}] " if emotion else ""
    prompt = (
        f"<|system|>\n{SYSTEM_PROMPT}\n"
        f"<|user|>\n{tag}{user_input}\n"
        f"<|assistant|>\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.3,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    for stop_token in ["<|user|>", "<|system|>", "<|assistant|>"]:
        if stop_token in response:
            response = response.split(stop_token)[0]

    return response.strip()



In [ ]:
def run_inference_tests(model, tokenizer):
    log.info("\n" + "="*60)
    log.info("ROUND 2 INFERENCE TESTS")
    log.info("="*60)

    test_cases = [
        ("I got promoted today, I can't believe it!",          "joy"),
        ("I've been feeling really lonely lately.",            "sadness"),
        ("My boss blamed me for something I didn't do!",       "anger"),
        ("I have a big exam tomorrow and I'm terrified.",      "fear"),
        ("I just bumped into my ex at the mall.",              "surprise"),
        ("I've been struggling with anxiety for months.",      "fear"),
        ("I feel hopeless and don't know what to do.",         "sadness"),
        ("I can't stop overthinking everything.",              "fear"),
        ("I feel like no one understands me.",                 "sadness"),
        ("What did you do over the weekend?",                   None),
    ]

    fp_words = ["i feel", "i felt", "i had", "i got", "i remember",
                "i also", "i went", "i once", "i know how", "i've been"]
    fp_count = 0

    for user_input, emotion in test_cases:
        response  = chat(model, tokenizer, user_input, emotion)
        is_fp     = any(w in response.lower() for w in fp_words)
        fp_count += int(is_fp)
        flag      = "⚠️  FP" if is_fp else "✅"

        log.info(f"\n{flag} [{emotion or 'no tag'}] {user_input}")
        log.info(f"🤖 {response}")
        log.info("-" * 60)

    fp_rate = fp_count / len(test_cases) * 100
    log.info(f"\n{'='*60}")
    log.info(f"First Person Rate : {fp_count}/{len(test_cases)} = {fp_rate:.1f}%")

    if fp_rate < 10:
        log.info("✅ First person FIXED — ready for edge deployment")
    elif fp_rate < 25:
        log.info("⚠️  Partially fixed — ship with system prompt guardrail")
    else:
        log.info("❌ Still present — consider more epochs")

In [ ]:

def main():
    log.info("=" * 60)
    log.info("TinyLlama Round 2 — MentalChat16K Fine-tuning")
    log.info("=" * 60)

    check_gpu()
    model, tokenizer     = load_round1_model()
    train_dataset, val_dataset = load_data(tokenizer)
    model                = apply_round2_lora(model)
    trainer              = train(model, tokenizer, train_dataset, val_dataset)
    save_and_merge(trainer.model, tokenizer)
    trainer.model.eval()
    run_inference_tests(trainer.model, tokenizer)

    log.info("\n" + "="*60)
    log.info("✅ Round 2 Complete!")
    log.info(f"Final model : {CONFIG['merged_model_dir']}")
    log.info(f"Logs        : {CONFIG['log_file']}")
    log.info("Next        : quantize to GGUF for edge deployment")
    log.info("="*60)


if __name__ == "__main__":
    main()